# HASTIKA @ ICON-2026 — Kaggle training notebook
**Settings (panel bên phải):** Accelerator = **GPU T4** · Internet = **On** · Persistence = *Files only* (nếu có).

Quy trình: 1) setup → 2) data → 3) TF-IDF → 4) transformers → 5) evaluate → 6) submission.
Toàn bộ kết quả nằm trong `/kaggle/working/hastika` và được lưu lại khi bạn **Save Version (Save & Run All)**.

> Code trong notebook được sinh tự động từ repo bằng `notebooks/build_notebook.py` — sửa code ở repo rồi build lại, đừng sửa trực tiếp ở đây.

In [ ]:
import os, subprocess, sys
WORK = '/kaggle/working/hastika' if os.path.exists('/kaggle') else os.path.abspath('hastika')
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
for d in ['configs', 'data/raw', 'data/processed', 'src/data', 'src/models', 'src/training',
          'src/evaluation', 'src/utils', 'checkpoints', 'logs', 'results']:
    os.makedirs(d, exist_ok=True)
print(WORK)

## 1) Source code

In [ ]:
%%writefile configs/base.yaml
# Shared defaults. Every model config starts with `_base_: base.yaml` and overrides what it needs.
# Any key can also be overridden from the CLI:  --set training.lr=1e-5 data.max_len=128
task: a                     # a = Hate / Non-Hate, b = 6 hate categories (overridable with --task)
seed: 42
run_name: null              # default: <config name>_<loss>_s<seed>

paths:
  raw_dir: data/raw         # organiser CSVs (+ *test*.csv once released)
  processed_dir: data/processed
  results_dir: results
  checkpoint_dir: checkpoints
  log_dir: logs

data:
  n_folds: 5
  fold_seed: 42             # keep fixed -> every model shares the same folds
  max_len: 96               # 95% of comments < 35 words

model:
  type: transformer         # transformer | tfidf
  name: google/muril-base-cased
  pooling: cls              # cls | mean
  dropout: 0.1

training:
  epochs: 4
  batch_size: 32
  eval_batch_size: 64
  grad_accum: 1
  lr: 2.0e-5
  head_lr: 1.0e-4           # classifier head is trained from scratch
  weight_decay: 0.01
  warmup_ratio: 0.1
  max_grad_norm: 1.0
  precision: auto           # auto | fp32 | fp16 | bf16
  num_workers: 2
  loss: auto                # auto (a: ce, b: wce) | ce | wce | focal
  class_weight: sqrt_inv    # none | inverse | sqrt_inv  (used by wce / focal)
  focal_gamma: 2.0
  label_smoothing: 0.0
  early_stopping_patience: 2

checkpoint:
  save: best                # best | none   (best = best epoch of every fold)
  half: true                # store weights in fp16 -> half the disk (~0.5 GB / fold for base models)


In [ ]:
%%writefile configs/bert.yaml
# Multilingual BERT — same family as the BERT baseline in the HASTIKA paper.
_base_: base.yaml
model:
  name: bert-base-multilingual-cased


In [ ]:
%%writefile configs/deberta.yaml
# mDeBERTa-v3 (multilingual DeBERTa). fp16 can overflow with DeBERTa-v3 -> use bf16 (A100/L4) or fp32 on T4/P100.
_base_: base.yaml
model:
  name: microsoft/mdeberta-v3-base
training:
  precision: fp32
  batch_size: 16
  grad_accum: 2


In [ ]:
%%writefile configs/indicbert.yaml
# IndicBERT v2 (AI4Bharat) — MLM + Samanantar + transliterated-TLM variant.
_base_: base.yaml
model:
  name: ai4bharat/IndicBERTv2-MLM-Sam-TLM


In [ ]:
%%writefile configs/modernbert.yaml
# ModernBERT is trained on English + code only -> expected to be weak on Kanglish.
# Kept as an ablation / reference point for the system paper. Needs transformers >= 4.48.
_base_: base.yaml
model:
  name: answerdotai/ModernBERT-base
  pooling: mean
training:
  lr: 3.0e-5


In [ ]:
%%writefile configs/muril.yaml
# MuRIL — pretrained on 17 Indian languages incl. transliterated (romanised) text. First choice for Kanglish.
_base_: base.yaml
model:
  name: google/muril-base-cased


In [ ]:
%%writefile configs/roberta.yaml
# XLM-RoBERTa (100 languages). For the large model use:
#   --set model.name=xlm-roberta-large training.lr=1e-5 training.batch_size=16 training.grad_accum=2
_base_: base.yaml
model:
  name: xlm-roberta-base


In [ ]:
%%writefile configs/tfidf.yaml
# B0 — TF-IDF (char_wb 2-5 + word 1-2) + linear classifier. CPU only, a few minutes.
_base_: base.yaml
model:
  type: tfidf
  clf: lr                   # lr | svm
  C_grid: [0.5, 1, 2, 4, 8, 16]   # best C chosen by OOF macro-F1 (svm: use smaller values, e.g. [0.05, 0.1, 0.25, 0.5, 1])
  char_ngram: [2, 5]
  word_ngram: [1, 2]
  max_features: 300000
  class_weight: auto        # auto (a: none, b: balanced) | none | balanced
checkpoint:
  save: none


In [ ]:
%%writefile evaluate.py
#!/usr/bin/env python3
"""
Compare runs on their out-of-fold (OOF) predictions and evaluate a blend.

  python evaluate.py --task b                                   # all runs of task b
  python evaluate.py --task b --runs tfidf_lr muril_wce_s42     # selected runs + their average
  python evaluate.py --task b --runs tfidf_lr muril_wce_s42 --weights 0.3 0.7
  python evaluate.py --task b --runs tfidf_lr muril_wce_s42 --optimize   # search blend weights on OOF

Writes per-class reports + confusion matrices to results/{task}/{run}/ (and results/{task}/_blend/),
and refreshes results/metrics.csv.
"""
import argparse
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.data.dataset import label_names, load_split
from src.evaluation.metrics import compute_metrics, per_class_report, plot_confusion, rebuild_metrics_table
from src.utils.config import load_config


def blend(probs, weights):
    w = np.asarray(weights, float); w = w / w.sum()
    return sum(wi * p for wi, p in zip(w, probs))


def optimize_weights(probs, y, step=0.1):
    grid = np.round(np.arange(0, 1 + 1e-9, step), 3)
    best = (None, -1)
    for w in itertools.product(grid, repeat=len(probs)):
        if abs(sum(w) - 1) > 1e-6:
            continue
        f1 = compute_metrics(y, blend(probs, w).argmax(1))["macro_f1"]
        if f1 > best[1]:
            best = (list(w), f1)
    return best


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", choices=["a", "b"], required=True)
    ap.add_argument("--runs", nargs="*")
    ap.add_argument("--weights", nargs="*", type=float)
    ap.add_argument("--optimize", action="store_true", help="grid-search blend weights (step 0.1) on OOF")
    ap.add_argument("--config", default="configs/base.yaml")
    a = ap.parse_args()

    cfg = load_config(a.config, task=a.task)
    res = Path(cfg["paths"]["results_dir"]) / a.task
    train = load_split(cfg, "train")
    labels = label_names(a.task)
    runs = a.runs or sorted(p.name for p in res.iterdir() if (p / "oof.npy").exists())
    if not runs:
        raise SystemExit(f"no finished runs in {res}")

    probs, rows = [], []
    for r in runs:
        p = np.load(res / r / "oof.npy"); probs.append(p)
        pred = p.argmax(1)
        m = compute_metrics(train.y, pred)
        rep = per_class_report(train.y, pred, labels)
        rep.to_csv(res / r / "per_class.csv")
        plot_confusion(train.y, pred, labels, res / r / "confusion.png", f"{a.task}/{r}  macro-F1 {m['macro_f1']:.4f}")
        rows.append({"run": r, **m, **{f"f1_{l}": rep.loc[l, "f1-score"] for l in labels}})

    if len(runs) > 1:
        if a.optimize:
            w, _ = optimize_weights(probs, train.y)
            print(f"optimized weights (OOF): {dict(zip(runs, w))}")
        else:
            w = a.weights or [1.0] * len(runs)
        pb = blend(probs, w); pred = pb.argmax(1)
        m = compute_metrics(train.y, pred)
        rep = per_class_report(train.y, pred, labels)
        out = res / "_blend"; out.mkdir(exist_ok=True)
        rep.to_csv(out / "per_class.csv")
        plot_confusion(train.y, pred, labels, out / "confusion.png", f"{a.task} blend  macro-F1 {m['macro_f1']:.4f}")
        json.dump({"runs": runs, "weights": [float(x) for x in w], **m}, open(out / "blend.json", "w"), indent=1)
        rows.append({"run": "BLEND(" + ", ".join(f"{r}:{x:.2f}" for r, x in zip(runs, np.asarray(w) / np.sum(w))) + ")",
                     **m, **{f"f1_{l}": rep.loc[l, "f1-score"] for l in labels}})

    pd.set_option("display.width", 200)
    print(pd.DataFrame(rows).set_index("run").round(4).to_string())
    rebuild_metrics_table(Path(cfg["paths"]["results_dir"]))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile inference.py
#!/usr/bin/env python3
"""
Create a CodaBench submission (predictions.csv + flat submission.zip).

Mode 1 — from saved probabilities of finished runs (single run or blend):
  python inference.py --task a --runs muril_ce_s42 --split val
  python inference.py --task b --runs tfidf_lr muril_wce_s42 roberta_wce_s42 --weights 1 2 2 --split test

Mode 2 — from checkpoints on any CSV (id + Comment), averaging all folds of each run.
Use this when the test file arrives after training (no retraining needed):
  python inference.py --task a --checkpoints checkpoints/a/muril_ce_s42 --input data/raw/binary_test_inputs.csv
  python inference.py --task b --checkpoints checkpoints/b/muril_wce_s42 checkpoints/b/roberta_wce_s42 \
                      --input data/raw/multiclass_test_inputs.csv --tfidf_runs tfidf_lr

Output: results/submissions/{task}_{split|input}_{tag}/predictions.csv + submission.zip
"""
import argparse
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from src.data.dataset import label_names, load_split
from src.data.preprocessing import read_inputs
from src.evaluation.metrics import compute_metrics
from src.utils.config import load_config


def predict_checkpoint_dir(run_ckpt: Path, texts, batch_size=64, precision="auto"):
    import torch
    from src.data.dataset import make_loader
    from src.models.factory import load_from_checkpoint
    from src.training.trainer import predict_proba, resolve_precision

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = resolve_precision(precision, device)
    folds = sorted(p for p in run_ckpt.glob("fold*") if (p / "model.pt").exists())
    if not folds:
        raise SystemExit(f"no fold checkpoints in {run_ckpt}")
    out = 0
    for fd in folds:
        model, tok, meta = load_from_checkpoint(fd)
        cfg = {"data": {"max_len": meta.get("max_len", 96)},
               "training": {"batch_size": batch_size, "eval_batch_size": batch_size, "num_workers": 2}}
        dl = make_loader(list(texts), None, tok, cfg, train=False)
        out = out + predict_proba(model.to(device), dl, device, amp) / len(folds)
        print(f"  {fd}: done (fold macro-F1 {meta.get('macro_f1')})")
        del model
    return out


def write_submission(ids, probs, labels, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    sub = pd.DataFrame({"id": ids, "label": np.array(labels)[probs.argmax(1)]})
    sub.to_csv(out_dir / "predictions.csv", index=False, encoding="utf-8")
    np.save(out_dir / "probs.npy", probs)
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "predictions.csv", arcname="predictions.csv")
    print(sub.label.value_counts().to_string())
    print(f"==> {out_dir / 'submission.zip'}  ({len(sub)} rows)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", choices=["a", "b"], required=True)
    ap.add_argument("--runs", nargs="*", default=[], help="finished runs in results/{task}/ (mode 1)")
    ap.add_argument("--split", choices=["val", "test"], default="val")
    ap.add_argument("--checkpoints", nargs="*", default=[], help="checkpoints/{task}/{run} folders (mode 2)")
    ap.add_argument("--input", help="CSV to predict in mode 2")
    ap.add_argument("--tfidf_runs", nargs="*", default=[],
                    help="mode 2: add TF-IDF runs (their {split}.npy must match --input; use --split)")
    ap.add_argument("--weights", nargs="*", type=float)
    ap.add_argument("--tag")
    ap.add_argument("--config", default="configs/base.yaml")
    a = ap.parse_args()

    cfg = load_config(a.config, task=a.task)
    labels = label_names(a.task)
    res = Path(cfg["paths"]["results_dir"])
    probs, names = [], []

    if a.checkpoints:                                            # ---- mode 2
        if not a.input:
            raise SystemExit("--input is required with --checkpoints")
        inp = read_inputs(a.input)
        for c in a.checkpoints:
            print(f"predicting with {c}")
            probs.append(predict_checkpoint_dir(Path(c), inp.text)); names.append(Path(c).name)
        for r in a.tfidf_runs:
            p = np.load(res / a.task / r / f"{a.split}.npy")
            assert len(p) == len(inp), f"{r}/{a.split}.npy has {len(p)} rows, input has {len(inp)}"
            probs.append(p); names.append(r)
        split_name = Path(a.input).stem
    else:                                                        # ---- mode 1
        if not a.runs:
            raise SystemExit("give --runs (mode 1) or --checkpoints + --input (mode 2)")
        inp = load_split(cfg, a.split)
        if inp is None:
            raise SystemExit(f"data/processed/{a.task}_{a.split}.csv not found — run src.data.preprocessing")
        train = load_split(cfg, "train")
        for r in a.runs:
            f = res / a.task / r / f"{a.split}.npy"
            if not f.exists():
                raise SystemExit(f"{f} missing (run trained before the {a.split} file existed?) -> use mode 2")
            probs.append(np.load(f)); names.append(r)
            oof = np.load(res / a.task / r / "oof.npy")
            print(f"  {r:35s} OOF {compute_metrics(train.y, oof.argmax(1))}")
        split_name = a.split

    w = np.asarray(a.weights or [1.0] * len(probs), float); w /= w.sum()
    if not a.checkpoints and len(a.runs) > 1:
        oof = sum(wi * np.load(res / a.task / r / "oof.npy") for wi, r in zip(w, a.runs))
        print(f"  {'BLEND':35s} OOF {compute_metrics(train.y, oof.argmax(1))}")
    p = sum(wi * pi for wi, pi in zip(w, probs))
    assert len(p) == len(inp)
    tag = a.tag or "+".join(names)[:80]
    write_submission(inp.id.values, p, labels, res / "submissions" / f"{a.task}_{split_name}_{tag}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile requirements.txt
pandas>=2.0
numpy
scipy
scikit-learn>=1.3
matplotlib
pyyaml
ftfy>=6.1
torch>=2.1
transformers>=4.48    # ModernBERT needs >= 4.48
sentencepiece         # MuRIL / XLM-R / mDeBERTa tokenizers
protobuf
tiktoken              # some fast-tokenizer conversions (mDeBERTa) need it


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/data/__init__.py


In [ ]:
%%writefile src/data/dataset.py
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

from src.data.preprocessing import TASKS


def load_split(cfg, split: str):
    """split: train | val | test. Returns None for a missing test file."""
    p = Path(cfg["paths"]["processed_dir"]) / f"{cfg['task']}_{split}.csv"
    if not p.exists():
        if split == "test":
            return None
        raise FileNotFoundError(p)
    return pd.read_csv(p, keep_default_na=False)


def label_names(task: str):
    return TASKS[task]["labels"]


class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], (None if self.labels is None else self.labels[i])


class Collator:
    def __init__(self, tokenizer, max_len: int):
        self.tok, self.max_len = tokenizer, max_len

    def __call__(self, batch):
        enc = self.tok([b[0] for b in batch], truncation=True, max_length=self.max_len,
                       padding=True, return_tensors="pt")
        if batch[0][1] is not None:
            enc["labels"] = torch.tensor([b[1] for b in batch], dtype=torch.long)
        return enc


def make_loader(texts, labels, tokenizer, cfg, train: bool):
    t = cfg["training"]
    return DataLoader(
        TextDataset(texts, labels),
        batch_size=t["batch_size"] if train else t["eval_batch_size"],
        shuffle=train,
        collate_fn=Collator(tokenizer, cfg["data"]["max_len"]),
        num_workers=t.get("num_workers", 2),
        pin_memory=torch.cuda.is_available(),
    )


In [ ]:
%%writefile src/data/preprocessing.py
"""
Text cleaning + fixed CV folds.

  python -m src.data.preprocessing                # uses configs/base.yaml paths

Writes {processed_dir}/{task}_train.csv (id, text, label, y, group, fold),
{task}_val.csv and, when a test file exists in raw_dir, {task}_test.csv.
Re-run after the test inputs are released: folds stay identical (same fold_seed).
"""
import argparse
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

TASKS = {
    "a": {
        "labels": ["Non-Hate", "Hate"],
        "train": "binary_train.csv", "val": "binary_validation_inputs.csv",
        "test_glob": "binary*test*.csv", "label_col": "Label",
    },
    "b": {
        "labels": ["Gender", "Political", "Religion", "Geo-political", "Violence", "Others"],
        "train": "multiclass_train.csv", "val": "multiclass_validation_inputs.csv",
        "test_glob": "multiclass*test*.csv", "label_col": "Hate Category",
    },
}

EMOJI = re.compile("[\U0001F000-\U0001FAFF☀-➿️]")
URL = re.compile(r"https?://\S+|www\.\S+")


def clean_text(t) -> str:
    import ftfy
    t = ftfy.fix_text(str(t))                       # repair mojibake (~14% of rows)
    t = html.unescape(t)
    t = re.sub(r"<br\s*/?>", " ", t, flags=re.I)
    t = re.sub(r"<[^>]+>", " ", t)
    t = URL.sub(" URL ", t)
    t = re.sub(r"(.)\1{3,}", r"\1\1\1", t)          # "thuuuuu" -> "thuuu"
    t = unicodedata.normalize("NFC", t)
    return re.sub(r"\s+", " ", t).strip()


def dedup_key(t: str) -> str:
    t = EMOJI.sub("", t.lower())
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return re.sub(r"[^\w\s]", "", t).strip()


def _find(raw_dir: Path, name: str) -> Path:
    for d in (raw_dir, raw_dir.parent):             # also accept the organiser layout data/*.csv
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found in {raw_dir}")


def read_inputs(path) -> pd.DataFrame:
    """Any CSV with an id column and a Comment/text column -> (id, text)."""
    df = pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)
    id_col = next(c for c in df.columns if c.strip().lower() in ("id", "index"))
    txt_col = next(c for c in df.columns if c.strip().lower() in ("comment", "text"))
    return pd.DataFrame({"id": df[id_col], "text": df[txt_col].map(clean_text)})


def prepare_task(task: str, raw_dir, processed_dir, n_folds=5, fold_seed=42, log=print):
    spec = TASKS[task]
    raw_dir, processed_dir = Path(raw_dir), Path(processed_dir)
    processed_dir.mkdir(parents=True, exist_ok=True)

    raw = pd.read_csv(_find(raw_dir, spec["train"]), encoding="utf-8-sig", keep_default_na=False)
    df = pd.DataFrame({"id": raw["id"], "text": raw["Comment"].map(clean_text),
                       "label": raw[spec["label_col"]].str.strip()})
    unknown = set(df.label) - set(spec["labels"])
    assert not unknown, f"unknown labels {unknown}"
    df["group"] = df.text.map(dedup_key)

    # duplicates: drop groups with conflicting labels, keep one row otherwise
    n0 = len(df)
    n_lab = df.groupby("group").label.transform("nunique")
    n_conflict = int((n_lab > 1).sum())
    df = df[n_lab == 1].drop_duplicates("group").reset_index(drop=True)
    df["y"] = df.label.map({l: i for i, l in enumerate(spec["labels"])})

    df["fold"] = -1
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=fold_seed)
    for k, (_, idx) in enumerate(sgkf.split(df, df.y, groups=df.group)):
        df.loc[idx, "fold"] = k
    df.to_csv(processed_dir / f"{task}_train.csv", index=False)

    val = read_inputs(_find(raw_dir, spec["val"]))
    val.to_csv(processed_dir / f"{task}_val.csv", index=False)

    msg = ""
    tests = sorted(raw_dir.glob(spec["test_glob"])) or sorted(raw_dir.parent.glob(spec["test_glob"]))
    if tests:
        test = read_inputs(tests[0])
        test.to_csv(processed_dir / f"{task}_test.csv", index=False)
        msg = f", test={len(test)} ({tests[0].name})"
    log(f"[task {task}] train {n0} -> {len(df)} (dropped {n_conflict} conflicting + "
        f"{n0 - n_conflict - len(df)} duplicate rows), val={len(val)}{msg}")
    return df


def ensure_processed(cfg, log=print):
    p = Path(cfg["paths"]["processed_dir"])
    task = cfg["task"]
    if not (p / f"{task}_train.csv").exists():
        prepare_task(task, cfg["paths"]["raw_dir"], p, cfg["data"]["n_folds"], cfg["data"]["fold_seed"], log)


def main():
    from src.utils.config import load_config
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/base.yaml")
    ap.add_argument("--tasks", nargs="+", default=["a", "b"])
    a = ap.parse_args()
    cfg = load_config(a.config)
    for t in a.tasks:
        df = prepare_task(t, cfg["paths"]["raw_dir"], cfg["paths"]["processed_dir"],
                          cfg["data"]["n_folds"], cfg["data"]["fold_seed"])
        print(pd.crosstab(df.fold, df.label).to_string(), "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluation/__init__.py


In [ ]:
%%writefile src/evaluation/metrics.py
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


def compute_metrics(y_true, y_pred) -> dict:
    return {"macro_f1": round(float(f1_score(y_true, y_pred, average="macro")), 4),
            "accuracy": round(float(accuracy_score(y_true, y_pred)), 4)}


def per_class_report(y_true, y_pred, labels) -> pd.DataFrame:
    r = classification_report(y_true, y_pred, labels=list(range(len(labels))), target_names=labels,
                              output_dict=True, zero_division=0)
    return pd.DataFrame(r).T.round(4)


def plot_confusion(y_true, y_pred, labels, out_path, title=""):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))), normalize="true")
    fig, ax = plt.subplots(figsize=(1.1 * len(labels) + 2, 1.0 * len(labels) + 1.5))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] > .55 else "#0b0b0b")
    ax.set_xticks(range(len(labels)), labels, rotation=35, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, loc="left", fontweight="bold")
    fig.tight_layout(); fig.savefig(out_path, dpi=130); plt.close(fig)


def summarize_folds(fold_f1) -> dict:
    return {"fold_f1": [round(float(f), 4) for f in fold_f1],
            "fold_f1_mean": round(float(np.mean(fold_f1)), 4),
            "fold_f1_std": round(float(np.std(fold_f1)), 4)}


def rebuild_metrics_table(results_dir) -> pd.DataFrame:
    """Collect results/{task}/{run}/metrics.json -> results/metrics.csv (one row per run)."""
    results_dir = Path(results_dir)
    rows = []
    for f in sorted(results_dir.glob("*/*/metrics.json")):
        m = json.load(open(f))
        rows.append({"task": f.parent.parent.name, "run": f.parent.name,
                     **{k: m.get(k) for k in ("macro_f1", "accuracy", "fold_f1_mean", "fold_f1_std",
                                              "model", "loss", "best_epochs", "has_test")}})
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["task", "macro_f1"], ascending=[True, False])
    df.to_csv(results_dir / "metrics.csv", index=False)
    return df


In [ ]:
%%writefile src/models/__init__.py


In [ ]:
%%writefile src/models/classifier.py
import json
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel


class TransformerClassifier(nn.Module):
    """Pretrained encoder + pooling + dropout + linear head."""

    def __init__(self, backbone_name: str, num_labels: int, pooling: str = "cls",
                 dropout: float = 0.1, backbone_config=None):
        super().__init__()
        if backbone_config is None:                      # training: load pretrained weights
            self.backbone = AutoModel.from_pretrained(backbone_name)
        else:                                            # inference: architecture only, weights come from ckpt
            self.backbone = AutoModel.from_config(backbone_config)
        self.meta = {"backbone_name": backbone_name, "num_labels": num_labels,
                     "pooling": pooling, "dropout": dropout}
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.backbone.config.hidden_size, num_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask, token_type_ids=None, **_):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        h = self.backbone(**kw).last_hidden_state
        if self.pooling == "mean":
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)
        else:
            pooled = h[:, 0]
        return self.head(self.dropout(pooled))

    def param_groups(self, lr, head_lr, weight_decay):
        no_decay = ("bias", "LayerNorm.weight", "layer_norm", "layernorm", "norm.weight")
        groups = {}
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            is_head = n.startswith("head.")
            wd = 0.0 if any(nd in n for nd in no_decay) else weight_decay
            key = (is_head, wd)
            groups.setdefault(key, {"params": [], "weight_decay": wd, "lr": (head_lr or lr) if is_head else lr})
            groups[key]["params"].append(p)
        return list(groups.values())

    # ---------- checkpoint I/O ----------
    def save(self, out_dir, tokenizer=None, half=True, extra=None):
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        sd = {k: (v.half() if half and v.is_floating_point() else v).cpu() for k, v in self.state_dict().items()}
        torch.save(sd, out_dir / "model.pt")
        self.backbone.config.save_pretrained(out_dir / "backbone")
        if tokenizer is not None:
            tokenizer.save_pretrained(out_dir / "tokenizer")
        json.dump({**self.meta, **(extra or {})}, open(out_dir / "meta.json", "w"), indent=1)

    @classmethod
    def load(cls, ckpt_dir, map_location="cpu"):
        ckpt_dir = Path(ckpt_dir)
        meta = json.load(open(ckpt_dir / "meta.json"))
        bcfg = AutoConfig.from_pretrained(ckpt_dir / "backbone")
        model = cls(meta["backbone_name"], meta["num_labels"], meta["pooling"], meta["dropout"],
                    backbone_config=bcfg)
        sd = torch.load(ckpt_dir / "model.pt", map_location=map_location)
        model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return model, meta


In [ ]:
%%writefile src/models/factory.py
from pathlib import Path

from transformers import AutoTokenizer

from src.models.classifier import TransformerClassifier


def build_tokenizer(cfg_or_name):
    name = cfg_or_name if isinstance(cfg_or_name, str) else cfg_or_name["model"]["name"]
    return AutoTokenizer.from_pretrained(name)


def build_model(cfg, num_labels: int):
    m = cfg["model"]
    if m["type"] != "transformer":
        raise ValueError(f"build_model only handles transformers, got {m['type']} (tfidf -> src.models.tfidf)")
    return TransformerClassifier(m["name"], num_labels, m.get("pooling", "cls"), m.get("dropout", 0.1))


def load_from_checkpoint(ckpt_dir):
    """-> (model, tokenizer, meta) from a folder written by TransformerClassifier.save()."""
    ckpt_dir = Path(ckpt_dir)
    model, meta = TransformerClassifier.load(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(ckpt_dir / "tokenizer")
    return model, tok, meta


In [ ]:
%%writefile src/models/tfidf.py
"""B0 — TF-IDF (char + word n-grams) + linear classifier, cross-validated on the fixed folds."""
import numpy as np
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.svm import LinearSVC

from src.evaluation.metrics import compute_metrics


def build_tfidf(mcfg: dict, C: float, balanced: bool):
    feats = FeatureUnion([
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=tuple(mcfg.get("char_ngram", [2, 5])),
                                 min_df=2, sublinear_tf=True, max_features=mcfg.get("max_features"))),
        ("word", TfidfVectorizer(analyzer="word", ngram_range=tuple(mcfg.get("word_ngram", [1, 2])),
                                 sublinear_tf=True, token_pattern=r"(?u)\b\w+\b|[\U0001F000-\U0001FAFF]")),
    ])
    cw = "balanced" if balanced else None
    clf = (LogisticRegression(C=C, max_iter=5000, class_weight=cw) if mcfg.get("clf", "lr") == "lr"
           else LinearSVC(C=C, class_weight=cw))
    return make_pipeline(feats, clf)


def _proba(pipe, texts):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(texts)
    d = pipe.decision_function(texts)
    if d.ndim == 1:
        d = np.stack([-d, d], 1)
    return softmax(d * 2.0, axis=1)      # pseudo-probabilities, only used for ensembling


def cross_validate(cfg, train, val, test, n_labels, log=print):
    mcfg = cfg["model"]
    cw = mcfg.get("class_weight", "auto")
    balanced = (cfg["task"] == "b") if cw == "auto" else cw == "balanced"
    folds = sorted(int(f) for f in train.fold.unique())
    best = None
    for C in mcfg.get("C_grid", [mcfg.get("C", 1.0)]):
        oof = np.zeros((len(train), n_labels)); pv = np.zeros((len(val), n_labels))
        pt = None if test is None else np.zeros((len(test), n_labels))
        fold_f1 = []
        for k in folds:
            tr, va = train[train.fold != k], train[train.fold == k]
            pipe = build_tfidf(mcfg, C, balanced).fit(tr.text, tr.y)
            oof[va.index] = _proba(pipe, va.text)
            fold_f1.append(compute_metrics(va.y, oof[va.index].argmax(1))["macro_f1"])
            pv += _proba(pipe, val.text) / len(folds)
            if test is not None:
                pt += _proba(pipe, test.text) / len(folds)
        m = compute_metrics(train.y, oof.argmax(1))
        log(f"  C={C}: OOF macro-F1 {m['macro_f1']:.4f} acc {m['accuracy']:.4f} "
            f"(fold {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f})")
        if best is None or m["macro_f1"] > best["macro_f1"]:
            best = {"macro_f1": m["macro_f1"], "C": C, "oof": oof, "val": pv, "test": pt,
                    "fold_f1": fold_f1, "balanced": balanced}
    return best


In [ ]:
%%writefile src/training/__init__.py


In [ ]:
%%writefile src/training/losses.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def class_weights(counts, mode: str):
    counts = np.asarray(counts, dtype=float)
    if mode in (None, "none"):
        return None
    w = counts.sum() / (len(counts) * counts)
    if mode == "sqrt_inv":
        w = np.sqrt(w)
    elif mode != "inverse":
        raise ValueError(f"unknown class_weight mode {mode}")
    return torch.tensor(w, dtype=torch.float)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("weight", weight if weight is not None else None)

    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, weight=self.weight, reduction="none")
        pt = torch.exp(-F.cross_entropy(logits, y, reduction="none"))
        return ((1 - pt) ** self.gamma * ce).mean()


def build_loss(name: str, tcfg: dict, class_counts) -> nn.Module:
    """name: ce | wce | focal (already resolved from 'auto')."""
    ls = tcfg.get("label_smoothing", 0.0)
    if name == "ce":
        return nn.CrossEntropyLoss(label_smoothing=ls)
    w = class_weights(class_counts, tcfg.get("class_weight", "sqrt_inv"))
    if name == "wce":
        return nn.CrossEntropyLoss(weight=w, label_smoothing=ls)
    if name == "focal":
        return FocalLoss(tcfg.get("focal_gamma", 2.0), w)
    raise ValueError(f"unknown loss {name}")


In [ ]:
%%writefile src/training/trainer.py
"""Single-fold trainer: AdamW + linear warmup, AMP, grad accumulation, early stopping on macro-F1."""
import copy
import math
import time

import numpy as np
import torch
from transformers import get_linear_schedule_with_warmup

from src.data.dataset import make_loader
from src.evaluation.metrics import compute_metrics


def resolve_precision(name: str, device: torch.device):
    if device.type != "cuda" or name == "fp32":
        return None
    if name == "auto":
        native_bf16 = torch.cuda.get_device_capability(device)[0] >= 8     # Ampere+ (A100, L4, 30xx/40xx)
        return torch.bfloat16 if native_bf16 else torch.float16           # T4 / P100 -> fp16
    return {"fp16": torch.float16, "bf16": torch.bfloat16}[name]


@torch.no_grad()
def predict_proba(model, loader, device, amp_dtype=None) -> np.ndarray:
    model.eval()
    out = []
    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=device.type, dtype=amp_dtype or torch.float32, enabled=amp_dtype is not None):
            out.append(model(**batch).float().softmax(-1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, 0))


class Trainer:
    def __init__(self, cfg, model, tokenizer, loss_fn, device, logger):
        self.cfg, self.t = cfg, cfg["training"]
        self.model = model.to(device)
        self.tok = tokenizer
        self.loss_fn = loss_fn.to(device)
        self.device = device
        self.log = logger
        self.amp_dtype = resolve_precision(self.t.get("precision", "auto"), device)

    def fit(self, train_df, valid_df, tag=""):
        t = self.t
        dl_tr = make_loader(train_df.text.tolist(), train_df.y.tolist(), self.tok, self.cfg, train=True)
        dl_va = make_loader(valid_df.text.tolist(), None, self.tok, self.cfg, train=False)

        opt = torch.optim.AdamW(self.model.param_groups(t["lr"], t.get("head_lr"), t["weight_decay"]))
        steps = math.ceil(len(dl_tr) / t["grad_accum"]) * t["epochs"]
        sch = get_linear_schedule_with_warmup(opt, int(t["warmup_ratio"] * steps), steps)
        scaler = torch.amp.GradScaler(enabled=self.amp_dtype == torch.float16)
        patience = t.get("early_stopping_patience") or t["epochs"]

        best = {"f1": -1.0, "epoch": 0, "state": None, "oof": None}
        history, bad = [], 0
        for ep in range(1, t["epochs"] + 1):
            self.model.train(); t0 = time.time(); total = 0.0
            opt.zero_grad(set_to_none=True)
            for i, batch in enumerate(dl_tr):
                batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}
                y = batch.pop("labels")
                with torch.autocast(device_type=self.device.type, dtype=self.amp_dtype or torch.float32,
                                    enabled=self.amp_dtype is not None):
                    logits = self.model(**batch)
                loss = self.loss_fn(logits.float(), y) / t["grad_accum"]
                scaler.scale(loss).backward()
                total += loss.item() * t["grad_accum"]
                if (i + 1) % t["grad_accum"] == 0 or i + 1 == len(dl_tr):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), t["max_grad_norm"])
                    scaler.step(opt); scaler.update(); sch.step()
                    opt.zero_grad(set_to_none=True)

            p_va = predict_proba(self.model, dl_va, self.device, self.amp_dtype)
            m = compute_metrics(valid_df.y, p_va.argmax(1))
            history.append({"epoch": ep, "loss": round(total / len(dl_tr), 4), **m})
            improved = m["macro_f1"] > best["f1"]
            self.log.info(f"{tag} ep {ep}/{t['epochs']} loss {total / len(dl_tr):.4f} "
                          f"macro-F1 {m['macro_f1']:.4f} acc {m['accuracy']:.4f} "
                          f"({time.time() - t0:.0f}s){' *' if improved else ''}")
            if improved:
                bad = 0
                best = {"f1": m["macro_f1"], "epoch": ep, "oof": p_va,
                        "state": {k: v.detach().to("cpu", copy=True) for k, v in self.model.state_dict().items()}}
            else:
                bad += 1
                if bad >= patience:
                    self.log.info(f"{tag} early stop (no improvement for {patience} epochs)")
                    break

        self.model.load_state_dict(best["state"])      # restore best epoch
        best["history"] = history
        del best["state"], opt
        return best

    def predict(self, texts):
        if texts is None or len(texts) == 0:
            return None
        dl = make_loader(list(texts), None, self.tok, self.cfg, train=False)
        return predict_proba(self.model, dl, self.device, self.amp_dtype)


In [ ]:
%%writefile src/utils/__init__.py


In [ ]:
%%writefile src/utils/config.py
"""YAML config loading with `_base_` inheritance and `key.sub=value` CLI overrides."""
import copy
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]


def _deep_update(base: dict, new: dict) -> dict:
    out = copy.deepcopy(base)
    for k, v in (new or {}).items():
        out[k] = _deep_update(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def _load(path: Path) -> dict:
    cfg = yaml.safe_load(open(path, encoding="utf-8")) or {}
    base = cfg.pop("_base_", None)
    return _deep_update(_load(path.parent / base), cfg) if base else cfg


def _parse_value(v: str):
    val = yaml.safe_load(v)
    if isinstance(val, str):                 # YAML 1.1 reads "1e-3" as a string
        try:
            val = float(val)
        except ValueError:
            pass
    return val


def set_by_dotted(cfg: dict, key: str, value):
    node = cfg
    *parents, last = key.split(".")
    for p in parents:
        node = node.setdefault(p, {})
    node[last] = value


def load_config(path, overrides=(), **top_level) -> dict:
    """overrides: ["training.lr=1e-5", ...]; top_level: task="b", seed=1 (None values ignored)."""
    path = Path(path)
    if not path.is_absolute() and not path.exists():
        path = ROOT / path
    cfg = _load(path)
    cfg["config_name"] = path.stem
    for o in overrides or ():
        k, v = o.split("=", 1)
        set_by_dotted(cfg, k.strip(), _parse_value(v))
    for k, v in top_level.items():
        if v is not None:
            cfg[k] = v
    # resolve paths relative to the project root
    cfg["paths"] = {k: str((ROOT / v) if not Path(v).is_absolute() else Path(v)) for k, v in cfg["paths"].items()}
    return cfg


def resolve_loss(cfg: dict) -> str:
    loss = cfg["training"].get("loss", "auto")
    return ("ce" if cfg["task"] == "a" else "wce") if loss == "auto" else loss


def run_name(cfg: dict) -> str:
    if cfg.get("run_name"):
        return cfg["run_name"]
    if cfg["model"]["type"] == "tfidf":
        return f"{cfg['config_name']}_{cfg['model'].get('clf', 'lr')}"
    return f"{cfg['config_name']}_{resolve_loss(cfg)}_s{cfg['seed']}"


def dump(cfg: dict, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    yaml.safe_dump(cfg, open(path, "w", encoding="utf-8"), sort_keys=False, allow_unicode=True)


# keys that do not change the model's results -> ignored when checking a resumed run
_VOLATILE = {"paths", "run_name", "config_name"}


def training_signature(cfg: dict) -> dict:
    sig = {k: v for k, v in cfg.items() if k not in _VOLATILE}
    sig = copy.deepcopy(sig)
    sig.get("training", {}).pop("num_workers", None)
    sig.get("training", {}).pop("eval_batch_size", None)
    sig.pop("checkpoint", None)
    return sig


In [ ]:
%%writefile src/utils/logger.py
import logging
import sys
from pathlib import Path


def get_logger(name: str = "hastika", log_file=None, level=logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S")
    if not any(isinstance(h, logging.StreamHandler) and not isinstance(h, logging.FileHandler)
               for h in logger.handlers):
        sh = logging.StreamHandler(sys.stdout); sh.setFormatter(fmt); logger.addHandler(sh)
    if log_file:
        log_file = str(Path(log_file).resolve())
        if not any(isinstance(h, logging.FileHandler) and h.baseFilename == log_file for h in logger.handlers):
            Path(log_file).parent.mkdir(parents=True, exist_ok=True)
            fh = logging.FileHandler(log_file, encoding="utf-8"); fh.setFormatter(fmt); logger.addHandler(fh)
    return logger


In [ ]:
%%writefile src/utils/seed.py
import os
import random

import numpy as np


def set_seed(seed: int, deterministic: bool = False):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


In [ ]:
%%writefile train.py
#!/usr/bin/env python3
"""
Cross-validated training on the fixed folds.

  python train.py --config configs/tfidf.yaml   --task a
  python train.py --config configs/muril.yaml   --task b
  python train.py --config configs/roberta.yaml --task b --set training.loss=focal --run_name xlmr_focal
  python train.py --config configs/muril.yaml   --task a --folds 0 1        # only some folds (resume later)

Outputs
  results/{task}/{run}/  oof.npy val.npy [test.npy] metrics.json config.yaml per-fold cache in folds/
  checkpoints/{task}/{run}/fold{k}/   (best epoch per fold, if checkpoint.save=best)
  logs/{task}_{run}.log ;  results/metrics.csv  (all runs)
Finished folds are cached: rerunning the same command resumes. Changing hyper-parameters under the
same run name is refused (use --run_name or --overwrite).
"""
import argparse
import json
import shutil
from pathlib import Path

import numpy as np
import yaml

from src.data.dataset import label_names, load_split
from src.data.preprocessing import ensure_processed
from src.evaluation.metrics import compute_metrics, rebuild_metrics_table, summarize_folds
from src.utils.config import dump, load_config, resolve_loss, run_name, training_signature
from src.utils.logger import get_logger
from src.utils.seed import set_seed


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--task", choices=["a", "b"])
    ap.add_argument("--seed", type=int)
    ap.add_argument("--run_name")
    ap.add_argument("--folds", type=int, nargs="*")
    ap.add_argument("--set", nargs="*", default=[], metavar="KEY=VALUE")
    ap.add_argument("--overwrite", action="store_true", help="delete previous results of this run")
    return ap.parse_args()


def check_run_dir(run_dir: Path, cfg, overwrite, log):
    cfg_file = run_dir / "config.yaml"
    if overwrite and run_dir.exists():
        shutil.rmtree(run_dir)
        ck = Path(cfg["paths"]["checkpoint_dir"]) / cfg["task"] / run_dir.name
        shutil.rmtree(ck, ignore_errors=True)
        log.info(f"removed previous results of {run_dir.name}")
    elif cfg_file.exists():
        old = yaml.safe_load(open(cfg_file, encoding="utf-8"))
        if training_signature(old) != training_signature(cfg):
            raise SystemExit(f"{run_dir} was trained with a different config. "
                             f"Use another --run_name or pass --overwrite.")
    dump(cfg, cfg_file)


def train_transformer(cfg, train, val, test, n_labels, run_dir, log):
    import torch
    from src.models.factory import build_model, build_tokenizer
    from src.training.losses import build_loss
    from src.training.trainer import Trainer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loss_name = resolve_loss(cfg)
    counts = np.bincount(train.y, minlength=n_labels)
    log.info(f"device={device} model={cfg['model']['name']} loss={loss_name} "
             f"precision={cfg['training']['precision']}")
    tokenizer = build_tokenizer(cfg)
    ck_root = Path(cfg["paths"]["checkpoint_dir"]) / cfg["task"] / run_dir.name
    cache = run_dir / "folds"; cache.mkdir(parents=True, exist_ok=True)

    all_folds = sorted(int(f) for f in train.fold.unique())
    for k in (cfg["_folds"] if cfg["_folds"] is not None else all_folds):
        f = cache / f"fold{k}.npz"
        if f.exists():
            log.info(f"fold {k}: cached -> skip"); continue
        set_seed(cfg["seed"] + k)
        tr, va = train[train.fold != k], train[train.fold == k]
        model = build_model(cfg, n_labels)
        trainer = Trainer(cfg, model, tokenizer, build_loss(loss_name, cfg["training"], counts), device, log)
        best = trainer.fit(tr, va, tag=f"[fold {k}]")
        p_val = trainer.predict(val.text)
        p_test = trainer.predict(test.text) if test is not None else None
        if cfg["checkpoint"]["save"] == "best":
            model.save(ck_root / f"fold{k}", tokenizer, half=cfg["checkpoint"].get("half", True),
                       extra={"fold": k, "epoch": best["epoch"], "macro_f1": best["f1"],
                              "task": cfg["task"], "labels": label_names(cfg["task"]),
                              "max_len": cfg["data"]["max_len"]})
        np.savez(f, oof=best["oof"], val=p_val, test=p_test if p_test is not None else np.empty(0),
                 f1=best["f1"], epoch=best["epoch"])
        json.dump(best["history"], open(cache / f"fold{k}_history.json", "w"), indent=1)
        log.info(f"fold {k}: best macro-F1 {best['f1']:.4f} @ epoch {best['epoch']}")
        del trainer, model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    missing = [k for k in all_folds if not (cache / f"fold{k}.npz").exists()]
    if missing:
        log.info(f"folds {missing} not trained yet -> rerun to aggregate"); return None

    oof = np.zeros((len(train), n_labels)); pv = np.zeros((len(val), n_labels))
    pt = np.zeros((len(test), n_labels)) if test is not None else None
    fold_f1, epochs = [], []
    for k in all_folds:
        z = np.load(cache / f"fold{k}.npz")
        oof[train.index[train.fold == k]] = z["oof"]
        pv += z["val"] / len(all_folds)
        if pt is not None:
            if z["test"].size == 0:
                log.warning(f"fold {k} was trained before the test file existed -> no test preds. "
                            f"Use inference.py --checkpoints (if checkpoints were saved) or retrain.")
                pt = None
            else:
                pt += z["test"] / len(all_folds)
        fold_f1.append(float(z["f1"])); epochs.append(int(z["epoch"]))
    return {"oof": oof, "val": pv, "test": pt, "fold_f1": fold_f1,
            "extra": {"model": cfg["model"]["name"], "loss": loss_name, "best_epochs": epochs}}


def main():
    a = parse()
    cfg = load_config(a.config, a.set, task=a.task, seed=a.seed, run_name=a.run_name)
    name = run_name(cfg)
    res_dir = Path(cfg["paths"]["results_dir"])
    run_dir = res_dir / cfg["task"] / name
    log = get_logger("hastika", Path(cfg["paths"]["log_dir"]) / f"{cfg['task']}_{name}.log")
    log.info(f"===== task {cfg['task']} | run {name} | config {a.config} =====")
    check_run_dir(run_dir, cfg, a.overwrite, log)
    cfg["_folds"] = a.folds

    ensure_processed(cfg, log.info)
    train, val, test = load_split(cfg, "train"), load_split(cfg, "val"), load_split(cfg, "test")
    labels = label_names(cfg["task"])
    set_seed(cfg["seed"])

    if cfg["model"]["type"] == "tfidf":
        from src.models.tfidf import cross_validate
        best = cross_validate(cfg, train, val, test, len(labels), log.info)
        out = {"oof": best["oof"], "val": best["val"], "test": best["test"], "fold_f1": best["fold_f1"],
               "extra": {"model": f"tfidf-{cfg['model'].get('clf', 'lr')} C={best['C']}",
                         "loss": "balanced" if best["balanced"] else "-", "best_C": best["C"]}}
    else:
        out = train_transformer(cfg, train, val, test, len(labels), run_dir, log)
        if out is None:
            return

    np.save(run_dir / "oof.npy", out["oof"]); np.save(run_dir / "val.npy", out["val"])
    if out["test"] is not None:
        np.save(run_dir / "test.npy", out["test"])
    metrics = {**compute_metrics(train.y, out["oof"].argmax(1)), **summarize_folds(out["fold_f1"]),
               **out["extra"], "has_test": out["test"] is not None, "labels": labels}
    json.dump(metrics, open(run_dir / "metrics.json", "w"), indent=1)
    rebuild_metrics_table(res_dir)
    log.info(f"==> {cfg['task']}/{name}: OOF macro-F1 {metrics['macro_f1']:.4f} | acc {metrics['accuracy']:.4f} "
             f"| folds {metrics['fold_f1_mean']:.4f} ± {metrics['fold_f1_std']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
!pip -q install ftfy sentencepiece tiktoken
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 2) Data
Lấy dữ liệu từ repo của ban tổ chức. **Khi test được phát hành (20/9):** chạy lại cell này (git pull) hoặc upload
file `*test*.csv` vào `data/raw/` (ví dụ từ một Kaggle Dataset: `!cp /kaggle/input/<dataset>/*test*.csv data/raw/`).

In [ ]:
if not os.path.exists('_organiser'):
    !git clone -q https://github.com/shankarb14/Hastika-ICON2026.git _organiser
else:
    !git -C _organiser pull -q
!cp _organiser/data/*.csv data/raw/
!ls data/raw
!python -m src.data.preprocessing

## 3) B0 — TF-IDF (CPU, vài phút)

In [ ]:
!python train.py --config configs/tfidf.yaml --task a
!python train.py --config configs/tfidf.yaml --task b

## 4) B1 — Transformers
Mỗi config × task ≈ 10–20 phút trên T4 (5 fold × 4 epoch, early stopping).
- Bị ngắt giữa chừng → chạy lại đúng lệnh, các fold đã xong được bỏ qua.
- Đổi siêu tham số → thêm `--run_name <tên mới>` (hoặc `--overwrite`).
- Mỗi run lưu 5 checkpoint fp16 (~0.5 GB/fold với model base). `/kaggle/working` giới hạn ~20 GB → xoá run không dùng (cell cuối mục này).

In [ ]:
CONFIGS = ['muril', 'roberta']            # thêm: 'indicbert', 'bert', 'deberta', 'modernbert'
TASKS = ['a', 'b']
for c in CONFIGS:
    for t in TASKS:
        !python train.py --config configs/{c}.yaml --task {t}

In [ ]:
# Ví dụ biến thể:
# !python train.py --config configs/muril.yaml --task b --set training.loss=focal
# !python train.py --config configs/muril.yaml --task a --seed 7
# !python train.py --config configs/roberta.yaml --task b --run_name xlmr_large_wce \
#       --set model.name=xlm-roberta-large training.lr=1e-5 training.batch_size=16 training.grad_accum=2

In [ ]:
!du -sh checkpoints/*/* 2>/dev/null; df -h /kaggle/working | tail -1
# xoá checkpoint của run không cần:  !rm -rf checkpoints/b/<run_name>

## 5) Evaluate (OOF)

In [ ]:
import pandas as pd
display(pd.read_csv('results/metrics.csv'))
!python evaluate.py --task a
!python evaluate.py --task b

In [ ]:
# blend + tối ưu trọng số trên OOF (đổi tên run theo bảng trên)
!python evaluate.py --task a --runs tfidf_lr muril_ce_s42 roberta_ce_s42 --optimize
!python evaluate.py --task b --runs tfidf_lr muril_wce_s42 roberta_wce_s42 --optimize

In [ ]:
from IPython.display import Image
Image('results/b/_blend/confusion.png')

## 6) Submission
- **Development phase (val):** mode 1 dùng xác suất đã lưu.
- **Evaluation phase (test):**
  - run train *sau* khi có test → `--split test`
  - run train *trước* khi có test → mode 2 dùng checkpoint (`--checkpoints ... --input ...`), không cần train lại.

In [ ]:
!python inference.py --task a --runs tfidf_lr muril_ce_s42 roberta_ce_s42 --split val --tag ens3
!python inference.py --task b --runs tfidf_lr muril_wce_s42 roberta_wce_s42 --split val --tag ens3
# test, từ checkpoint:
# !python inference.py --task b --checkpoints checkpoints/b/muril_wce_s42 checkpoints/b/roberta_wce_s42 \
#       --input data/raw/multiclass_test_inputs.csv --tag ens2
!find results/submissions -name submission.zip